# Descarga de datos geográficos de GeoElectoral

El Órgano Electoral mantiene datos geográficos públicos en una [instancia arcgis](https://geoelectoral.oep.org.bo/oep/rest/services/). En este notebook descargamos los campos más esenciales (información geográfica + identificadores) de capas de asientos y recintos en las elecciones subnacionales 2026 a documentos GeoPackage. Este cuaderno reutiliza [código que ya había escrito](https://github.com/datosbolivia/elecciones2025/blob/main/geo/geoelectoral.ipynb) para las elecciones nacionales de 2025.

In [1]:
import requests
import geopandas as gpd
from os import makedirs
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


Especificamos los datasets que queremos:

In [2]:
GEOELECTORAL = "https://geoelectoral.oep.org.bo/oep/rest/services"
DATA_PATHS = [
    # {
    #     "year": 2026,  # el año, que utilizaremos como el nombre del directorio donde guardamos los datos
    #     "name": "recintos",  # el nombre, que utilizaremos como el filename del geopackage
    #     "item": "AsientosElectorales/AsientosRecintos_22_01_2026",  # el item dentro del servidor arcgis
    #     "mapserver": 1,  # el mapserver dentro del item donde están los datos
    #     "fields": {
    #         "IdLoc": "asiento",
    #         "Reci": "recinto",
    #     },  # un diccionario que mapea los campos que queremos, aparte de la información geográfica, y los nombres con los que queremos almacenar esos campos
    # },
    # {
    #     "year": 2026,
    #     "name": "asientos",
    #     "item": "AsientosElectorales/AsientosRecintos_22_01_2026",
    #     "mapserver": 0,
    #     "fields": {"IdLoc": "asiento"},
    # },
    {
        "year": 2026,
        "name": "municipios",
        "item": "AsientosElectorales/AsientosRecintos_22_01_2026",
        "mapserver": 3,
        "fields": {"C_UT": "municipio"},
    },
]

Y los descargamos.

In [3]:

def _make_session():
    retry = Retry(
        total=5,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )
    session = requests.Session()
    session.headers.update({"User-Agent": "geoelectoral-downloader/1.0"})
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def _get_json(session, url, params=None, timeout=30):
    resp = session.get(url, params=params, timeout=timeout)
    resp.raise_for_status()
    return resp.json()


In [4]:

def descargar_capa(datapath, output_crs=4326, timeout=30):
    print(f"Descargando {datapath['name']} para {datapath['year']}...")
    directory = datapath["year"]
    filename = f"{datapath['name']}.gpkg"
    makedirs(str(directory), exist_ok=True)

    session = _make_session()

    mapserver = f"{GEOELECTORAL}/{datapath['item']}/MapServer/{datapath['mapserver']}"
    service_info = _get_json(session, f"{mapserver}?f=json", timeout=timeout)
    max_record_count = service_info.get("maxRecordCount", 1000)

    features = []
    features_per_call = min(100, max_record_count)
    fields = ",".join(datapath["fields"].keys())

    ids_payload = _get_json(
        session,
        f"{mapserver}/query",
        params={"where": "1=1", "returnIdsOnly": "true", "f": "json"},
        timeout=timeout,
    )
    feature_ids = ids_payload.get("objectIds") or []

    for i in tqdm(range(0, len(feature_ids), features_per_call)):
        payload = _get_json(
            session,
            f"{mapserver}/query",
            params={
                "objectIds": ",".join(map(str, feature_ids[i : i + features_per_call])),
                "outFields": fields,
                "returnGeometry": "true",
                "outSR": output_crs,
                "f": "geojson",
            },
            timeout=timeout,
        )
        features.extend(payload.get("features", []))

    gdf = gpd.GeoDataFrame.from_features(features, crs=output_crs).rename(
        columns=datapath["fields"]
    )
    if output_crs != 4326:
        gdf = gdf.to_crs(output_crs)
    gdf.to_file(f"{directory}/{filename}", driver="GPKG")


In [5]:
for datapath in DATA_PATHS:
    descargar_capa(datapath)

Descargando municipios para 2026...


100%|██████████| 4/4 [01:01<00:00, 15.47s/it]
